In [ ]:
import pandas as pd
import plotly.express as px
import json
import os

In [ ]:
location = pd.read_csv('data/taxi-zone-lookup.csv')
print('Lines / Columns : ', location.shape)

In [ ]:
files = ["data/uber-raw-data-apr14.csv", "data/uber-raw-data-may14.csv", "data/uber-raw-data-jun14.csv", 'data/uber-raw-data-jul14.csv', 'data/uber-raw-data-aug14.csv', 'data/uber-raw-data-sep14.csv']

raw_2014 = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print('Lines / Columns : ', raw_2014.shape)

In [ ]:
raw_2015 = pd.read_csv('data/uber-raw-data-janjune-15.csv')
print('Lines / Columns : ', raw_2015.shape)

In [ ]:
location = pd.read_csv('c/location.csv')
print(f'location.csv : {location.shape[0]} lines / {location.shape[1]} columns')
pickup_14 = pd.read_csv('c/2014.csv')
print(f'2014.csv : {pickup_14.shape[0]} lines / {pickup_14.shape[1]} columns')
pickup = pd.read_csv('c/2015.csv')
print(f'2015.csv : {pickup.shape[0]} lines / {pickup.shape[1]} columns')

with open("c/taxi_zones.geojson", "r", encoding="utf-8") as f:
    geojson = json.load(f)

s1 = pd.read_csv('c/s1.csv')
print(f's1.csv : {s1.shape[0]} lines / {s1.shape[1]} columns')
t1 = pd.read_csv('c/t1.csv')
print(f't1.csv : {t1.shape[0]} lines / {t1.shape[1]} columns')

In [ ]:
df = pd.read_csv('c/df.csv')
print(f'df.csv : {pickup.shape[0]} lines / {df.shape[1]} columns')

In [ ]:
df.head(1)

## Explore dataset

In [ ]:
dataframes = [location, raw_2014, raw_2015]
[print(f"Display of dataset: {display(df.head())}") for df in dataframes]

We observe :
- date time MM/DD/YYYY
- lowercase and uppercase
- sorted raw data 



In [ ]:
# Basic stats
dataframes = {'location':location, 'raw_2014':raw_2014, 'raw_2015':raw_2015}

for name, data in dataframes.items():
    print()
    print('-------------------Dataframe : ', name, '-------------------')
    print(f"Rows :  \t {data.shape[0]}")
    print(f"Columns : \t {data.shape[1]}")

    print()
    print("Basics statistics: ")
    data_desc = data.describe(include='all') # numeric variables and catégorielles
    display(data_desc)

    print("Percentage of missing values")
    data_missing_values = data.isnull().sum()/data.shape[0]*100
    print(f"{name} : \n{data_missing_values}")
print('Done...')

In [ ]:
# Lowercase dataframes
dataframes = {'location':location, 'raw_2014':raw_2014, 'raw_2015':raw_2015}
for name, data in dataframes.items():
    data.columns = data.columns.str.lower() # lowercase columns names
    for col in data.select_dtypes(include=["object", 'string']).columns: # avoid warning for multi execution
        data[col] = data[col].str.lower()
        print()
    print('-------------------',name,'-------------------')
    print(data.head(1))

## Identify base in 2014

In [ ]:
raw_2014.head(1)

In [ ]:
# raw_2014['base'].isin(raw_2015['dispatching_base_num']).value_counts()

On retrouve le nombre de lignes exact de raw_2014. Aucune base ne diffère des dispatching_base_num.
Pour s'en assurer, comparons les valeurs uniques.

In [ ]:
b14 = set(raw_2014['base'].unique())
b15_dispatch = set(raw_2015['dispatching_base_num'].unique())
b15_licence = set(raw_2015['affiliated_base_num'].unique())
print(len(b14), len(b15_dispatch), len(b15_licence), len(b14 & b15_dispatch & b15_licence))

Base correspond à la référence du centre de dispatch. De 2014 à 2015, le nombre des centres de dispatch a été augmenté de 3.

## Identify location pickup

locationid est présent sur uber-raw-data-janjune-15 et taxi-zone-lookup. Nous pouvons les fusionner à partir de cette colonne.

## Analyze one hour on one day

In [ ]:
raw_2015.head(1)

In [ ]:
raw_2014['date/time'].dtype

In [ ]:
raw_2015['pickup_date'].dtype

## Detect outlyers

On cherche seulement les points aberrants. Dans l'eau ou en dehors de l'état de new york.

## Search for duplicated values

There are no duplicated values, only values very similar. Des points proches signalent justement une forte densité de pickups, donc une hot-zone. Les supprimer ferait perdre cette information et fausserait les clusters (KMeans comme DBSCAN).

# Univariated analysis

## Analyze taxi_zone_lookup.csv

In [ ]:
count_location_id_per_zone = location['borough'].value_counts()
count_location_id_per_zone

In [ ]:
location.head(1)

In [ ]:
mask = location['locationid'].value_counts() == 1
print(mask.all()) # True attendu si toutes les valeurs sont uniques

In [ ]:
nb_borough = location['borough'].nunique()
nb_borough

In [ ]:
count_location_id_per_zone = location['borough'].unique()
count_location_id_per_zone

In [ ]:
location[location['borough'] == 'ewr']

We observe 5 boroughs in New York. EWR correspond à la locationid 1 : la zone Newark Airport. EWR est le code IATA de l’aéroport de Newark, situé dans le New Jersey. Tous les boroughs ne sont pas identifiés.

In [ ]:
location[location['borough'] == 'unknown']

In [ ]:
location['zone'].value_counts()

In [ ]:
location[location.duplicated("zone", keep=False)]  # Corona (56, 57), Governor's Island (103 à 105), Unknown (264, 265)

Borough est beaucoup moins précis que zone pour tracer les pickups. 
We observe :
- zone de corona dans le queens en doublon. 
- 3 références communes sur les 3 îlots sur l'Hudson.
- deux zones sans étiquette.

## Analyze raw_2014

In [ ]:
pts = s1.groupby(["lat", "lon"]).size().reset_index(name="nb_pickups")
fig = px.scatter_map(pts, lat="lat", lon="lon", size="nb_pickups",
                     color=pts["nb_pickups"].astype(str), labels={"color": "Nb pickups"}, # traiter les chiffres comme des catégories
                     size_max=8, zoom=10, map_style="carto-positron")
fig.show()

## Analyze raw_2015

Le fichier trace les pickups sur 6 mois d'exploitation en 2015 de janvier à juin. Il contient la datetime, les références du dispatch, taxi et emplacement. 
We observe missing values sur le dispatch 1,14 %.

In [ ]:
# pickup_14['base'].isin(pickup['dispatching_base_num']).value_counts()

In [ ]:
cle = [k for k in geojson["features"][0]["properties"] if k in ("locationid", "location_id")][0]
for feat in geojson["features"]:
    feat["properties"][cle] = feat["properties"][cle]
counts = t1["locationid"].value_counts().reset_index()
counts

In [ ]:
uniques = t1[["locationid", "borough"]].drop_duplicates()
uniques["locationid"] = uniques["locationid"].astype(str)

px.choropleth_map(uniques, geojson=geojson, locations="locationid",
                  featureidkey=f"properties.{cle}", color="borough",
                  zoom=10, map_style="carto-positron",
                  center={"lat": 40.73, "lon": -73.94}).show()

In [ ]:
px.choropleth_map(counts, geojson=geojson, locations="locationid",
                  featureidkey=f"properties.{cle}", color="count",
                  zoom=10, map_style="carto-positron",
                  center={"lat": 40.73, "lon": -73.94}).show()

## Identify location pickup

locationid est présent sur uber-raw-data-janjune-15 et taxi-zone-lookup. Nous pouvons les fusionner à partir de cette colonne.

## Analyze pickup_14

Les fichiers tracent 6 mois d'exploitation en 2014 d'avril à septembre. Les informations diffèrent : date/time, lat, lon, base.
Cette fois, nous disposons des coordonnées des pickups.
Nous devons identifier les emplacements et établir si base est la référence du dispatch ou du taxi. 

In [ ]:
pickup_14_volume = pickup_14["base"].value_counts()
pickup_volume = pickup["dispatching_base_num"].value_counts()
print(pickup_14_volume, pickup_volume)

In [ ]:
v = pd.concat([pickup_14["base"].value_counts().rename("2014"),
               pickup["dispatching_base_num"].value_counts().rename("2015")], axis=1).fillna(0)
v.index.name = "base"

fig = px.bar(v.reset_index(), x="base", y=["2014", "2015"], barmode="group",
             title="Evolution of volume pickups (millions) by dispatch base on 6 months",
             labels={"value": "Pickups", "variable": "Year"})
fig.update_layout(hovermode="x unified")
fig.show()

# Bivariated analysis

In [ ]:
a = pickup[pickup['locationid'] == 1]
a

In [ ]:
b = pickup[pickup['locationid'].isin([264, 265])]
b

## Identify base in 2015

In [ ]:
# pickup_14['base'].isin(raw_2015['dispatching_base_num']).value_counts()

On retrouve le nombre de lignes exact de raw_2014. Aucune base ne diffère des dispatching_base_num.
Pour s'en assurer, comparons les valeurs uniques.

In [ ]:
b14 = set(pickup_14['base'].unique())
b15_dispatch = set(pickup['dispatching_base_num'].unique())
b15_licence = set(pickup['affiliated_base_num'].unique())
print(len(b14), len(b15_dispatch), len(b15_licence), len(b14 & b15_dispatch & b15_licence))

Base correspond à la référence du centre de dispatch. De 2014 à 2015, le nombre des centres de dispatch a été augmenté de 3.

## Identify location pickup

locationid est présent sur uber-raw-data-janjune-15 et taxi-zone-lookup. Nous pouvons les fusionner à partir de cette colonne.

## Treat missing values

La référence du taxi est inutile pour kmeans. On remplace par une étiquette explicite.

In [ ]:
pickup['affiliated_base_num'] = pickup['affiliated_base_num'].fillna('unknown') # drop lines where values are missing. 
missing_values = pickup.isnull().sum()/pickup.shape[0]*100
print(missing_values)

## Identify base on raw_2014

Les fichiers tracent 6 mois d'exploitation en 2014 d'avril à septembre. Les informations diffèrent : date/time, lat, lon, base.
Cette fois, nous disposons des coordonnées des pickups.
Nous devons identifier les emplacements et établir si base est la référence du dispatch ou du taxi. 

In [ ]:
pickup_14_volume = pickup_14["base"].value_counts()
pickup_volume = pickup["dispatching_base_num"].value_counts()
print(pickup_14_volume, pickup_volume)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

v14 = pickup_14["base"].value_counts().rename("2014")
v15 = pickup["dispatching_base_num"].value_counts().rename("2015")

pd.concat([v14, v15], axis=1).plot(kind="bar", figsize=(8, 5))
plt.xlabel("dispatch_base")
plt.ylabel("pickups")
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

v = pd.concat([pickup_14["base"].value_counts().rename("2014"),
               pickup["dispatching_base_num"].value_counts().rename("2015")], axis=1)

fig = px.bar(v.reset_index(), x="index", y=["2014", "2015"], barmode="group",
             title="Nombre de pickups par base de dispatch",
             labels={"index": "Base", "value": "Pickups", "variable": "Year"})
fig.show()

## Analyze one hour on one day

In [ ]:
pickup.head(1)

In [ ]:
pickup_14['date/time'].dtype

In [ ]:
pickup['pickup_date'].dtype

In [ ]:
df.head()

In [ ]:
print(df.groupby('dispatching_base_num')[['lat', 'lon']].agg(['mean', 'std']))

In [ ]:
print(df.groupby('affiliated_base_num')[['lat', 'lon']].agg(['mean', 'std']))

In [ ]:
print(df['affiliated_base_num'].value_counts().tail(20))